# Task 2 - Model Building and Training
Training fraud detection models for both e-commerce and credit card datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, f1_score
)
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import joblib
import os
import sys
sys.path.insert(0, '..')
from src.data_processing import (
    load_data, clean_fraud_data, clean_credit_data,
    merge_geolocation, engineer_features,
    transform_features, handle_imbalance
)
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
print('Libraries loaded!')

## Helper Functions

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = {
        'model': model_name,
        'f1': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_test, y_prob), 4),
        'auc_pr': round(average_precision_score(y_test, y_prob), 4)
    }
    print(f'\n=== {model_name} ===')
    print(f'F1 Score:  {metrics["f1"]}')
    print(f'ROC-AUC:   {metrics["roc_auc"]}')
    print(f'AUC-PR:    {metrics["auc_pr"]}')
    print(classification_report(y_test, y_pred))
    return metrics


def plot_confusion_matrix(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Legit', 'Fraud'],
        yticklabels=['Legit', 'Fraud']
    )
    plt.title(f'Confusion Matrix - {title}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    fname = title.replace(' ', '_').lower()
    plt.savefig(f'../data/processed/cm_{fname}.png')
    plt.show()


print('Helper functions ready!')

## Load Data

In [ ]:
fraud_df, ip_df, credit_df = load_data()
print(f'Fraud_Data: {fraud_df.shape}')
print(f'IP_Data: {ip_df.shape}')
print(f'CreditCard: {credit_df.shape}')

## Dataset 1 - E-Commerce Fraud Detection

In [ ]:
fraud_df = clean_fraud_data(fraud_df)
fraud_df = merge_geolocation(fraud_df, ip_df)
fraud_df = engineer_features(fraud_df)
fraud_df = transform_features(fraud_df, 'class')
print(f'Processed shape: {fraud_df.shape}')
print(f'Class distribution:\n{fraud_df["class"].value_counts()}')

In [ ]:
X_f = fraud_df.drop(columns=['class'])
y_f = fraud_df['class']
X_f_train, X_f_test, y_f_train, y_f_test = train_test_split(
    X_f, y_f, test_size=0.2, random_state=42, stratify=y_f
)
print(f'Train: {X_f_train.shape[0]:,} | Test: {X_f_test.shape[0]:,}')
print(f'Before SMOTE: {y_f_train.value_counts().to_dict()}')
X_f_train, y_f_train = handle_imbalance(X_f_train, y_f_train)
print(f'After SMOTE: {pd.Series(y_f_train).value_counts().to_dict()}')

In [ ]:
mlflow.set_experiment('fraud_ecommerce')
results_fraud = []

with mlflow.start_run(run_name='LR_Fraud'):
    lr_f = LogisticRegression(
        class_weight='balanced', random_state=42, max_iter=1000
    )
    lr_f.fit(X_f_train, y_f_train)
    m = evaluate_model(lr_f, X_f_test, y_f_test, 'Logistic Regression - Fraud')
    results_fraud.append(m)
    plot_confusion_matrix(lr_f, X_f_test, y_f_test, 'LR Fraud')
    mlflow.log_metrics({'f1': m['f1'], 'roc_auc': m['roc_auc'], 'auc_pr': m['auc_pr']})
    mlflow.sklearn.log_model(lr_f, 'lr_fraud')
    joblib.dump(lr_f, '../models/lr_fraud.pkl')
    print('Logistic Regression trained and saved!')

In [ ]:
with mlflow.start_run(run_name='RF_Fraud'):
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [5, 10, None]
    }
    rf_base = RandomForestClassifier(
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf_f = GridSearchCV(
        rf_base, param_grid, cv=3,
        scoring='average_precision', n_jobs=-1
    )
    rf_f.fit(X_f_train, y_f_train)
    best_rf_f = rf_f.best_estimator_
    print(f'Best params: {rf_f.best_params_}')
    m = evaluate_model(best_rf_f, X_f_test, y_f_test, 'Random Forest - Fraud')
    results_fraud.append(m)
    plot_confusion_matrix(best_rf_f, X_f_test, y_f_test, 'RF Fraud')
    mlflow.log_params(rf_f.best_params_)
    mlflow.log_metrics({'f1': m['f1'], 'roc_auc': m['roc_auc'], 'auc_pr': m['auc_pr']})
    mlflow.sklearn.log_model(best_rf_f, 'rf_fraud')
    joblib.dump(best_rf_f, '../models/rf_fraud.pkl')
    print('Random Forest trained and saved!')

In [ ]:
print('\n=== FRAUD DATA - MODEL COMPARISON ===')
results_fraud_df = pd.DataFrame(results_fraud)
print(results_fraud_df.to_string(index=False))

## Dataset 2 - Credit Card Fraud Detection

In [ ]:
credit_df = clean_credit_data(credit_df)
X_c = credit_df.drop(columns=['Class'])
y_c = credit_df['Class']
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)
print(f'Train: {X_c_train.shape[0]:,} | Test: {X_c_test.shape[0]:,}')
print(f'Before SMOTE: {y_c_train.value_counts().to_dict()}')
X_c_train, y_c_train = handle_imbalance(X_c_train, y_c_train)
print(f'After SMOTE: {pd.Series(y_c_train).value_counts().to_dict()}')

In [ ]:
mlflow.set_experiment('fraud_creditcard')
results_credit = []

with mlflow.start_run(run_name='LR_Credit'):
    lr_c = LogisticRegression(
        class_weight='balanced', random_state=42, max_iter=1000
    )
    lr_c.fit(X_c_train, y_c_train)
    m = evaluate_model(lr_c, X_c_test, y_c_test, 'Logistic Regression - Credit')
    results_credit.append(m)
    plot_confusion_matrix(lr_c, X_c_test, y_c_test, 'LR Credit')
    mlflow.log_metrics({'f1': m['f1'], 'roc_auc': m['roc_auc'], 'auc_pr': m['auc_pr']})
    mlflow.sklearn.log_model(lr_c, 'lr_credit')
    joblib.dump(lr_c, '../models/lr_credit.pkl')
    print('Logistic Regression Credit trained and saved!')

In [ ]:
with mlflow.start_run(run_name='RF_Credit'):
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [5, 10, None]
    }
    rf_base_c = RandomForestClassifier(
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf_c = GridSearchCV(
        rf_base_c, param_grid, cv=3,
        scoring='average_precision', n_jobs=-1
    )
    rf_c.fit(X_c_train, y_c_train)
    best_rf_c = rf_c.best_estimator_
    print(f'Best params: {rf_c.best_params_}')
    m = evaluate_model(best_rf_c, X_c_test, y_c_test, 'Random Forest - Credit')
    results_credit.append(m)
    plot_confusion_matrix(best_rf_c, X_c_test, y_c_test, 'RF Credit')
    mlflow.log_params(rf_c.best_params_)
    mlflow.log_metrics({'f1': m['f1'], 'roc_auc': m['roc_auc'], 'auc_pr': m['auc_pr']})
    mlflow.sklearn.log_model(best_rf_c, 'rf_credit')
    joblib.dump(best_rf_c, '../models/rf_credit.pkl')
    print('Random Forest Credit trained and saved!')

In [ ]:
print('\n=== CREDIT CARD - MODEL COMPARISON ===')
results_credit_df = pd.DataFrame(results_credit)
print(results_credit_df.to_string(index=False))
print('\n=== ALL DONE! Models saved in models/ folder ===')